As we use preloaded dataset diabetes for demonstrating this there are no steps needed to preprocess the dataset. 
imports

In [32]:
from sklearn.datasets import load_diabetes
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split

In [ ]:
Getting Reference data using Ordinary Least Squares(OLS) method:

In [43]:
x,y = load_diabetes(return_X_y=True)
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=2)
lr=LinearRegression()
lr.fit(x_train,y_train)
print(lr.coef_)
print(lr.intercept_)
y_pred = lr.predict(x_test)
r2_score(y_test,y_pred)
#print("r2_score:",r2_score)

[  -9.15865318 -205.45432163  516.69374454  340.61999905 -895.5520019
  561.22067904  153.89310954  126.73139688  861.12700152   52.42112238]
151.88331005254167


0.439933866156897

Custom class for Batch GD implementation

In [44]:
class gdregressor:
    def __init__(self,learn_rate=0.01,epochs=100):
        self.lr=learn_rate
        self.epochs=epochs
        self.coef_=None
        self.intercept_=None

    def fit(self,x_train,y_train):
        #init the coefs and intercept
        self.intercept_=0
        self.coef_=np.ones(x_train.shape[1])

        for i in range(self.epochs):
            #Calculate y_hat for the initialized values of coefs and intercept
            y_hat=np.dot(x_train,self.coef_)+self.intercept_

            #update intercept
            di=-2 * np.mean(y_train - y_hat)
            self.intercept_=self.intercept_- (self.lr * di)

            #update coefs
            dc = -2 * np.dot((y_train - y_hat),x_train)/x_train.shape[0]
            self.coef_ = self.coef_ - (self.lr * dc)

        print(self.intercept_, self.coef_)

    def predict(self,x_test):
        return np.dot(x_test,self.coef_) + self.intercept_

In [45]:
gdr=gdregressor(epochs=50,learn_rate=0.01) #epochs=100, learn_rate=0.8836568
gdr.fit(x_train,y_train)
y_pred = gdr.predict(x_test)
r2_score(y_test,y_pred)

95.67680309579764 [ 1.8200381   1.05425059  2.9743748   2.65194416  1.69939102  1.44070863
 -0.17916682  2.22598299  3.11921427  2.23353436]


-0.7097974354984797

Problems with Batch GD:
1. Algorithm is very slow on big data due to computation complexity while calculating derivatives for intercept and coefs. Works well only for small sized datasets with less number of columns.
2. With big data due to high RAM consumption to load X_TRAIN dataset, the Vectorization is not possible to perform one shot matrix multiplication to find all coefs. This is a hardware limitation as well, as high hardware configuration becomes a pre-requisite for Batch GD.
3. Performs updates once per epoch processing all data.
4. Slower convergence

In [20]:
#Batch GD using sklearn SGDRegressor
from sklearn.datasets import load_diabetes
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
x,y=load_diabetes(return_X_y=True)
xs=StandardScaler().fit_transform(x)
xs_train,xs_test,y_train,y_test=train_test_split(xs,y,test_size=0.3,random_state=2)
sgd_batch=SGDRegressor(max_iter=500,learning_rate='constant',eta0=0.00015,random_state=0)
sgd_batch.fit(xs_train,y_train)
y_hat=sgd_batch.predict(xs_test)
print("Coefficients:",sgd_batch.coef_)
print("Intercept:",sgd_batch.intercept_)
print("Batch GD r2_score:",r2_score(y_test,y_hat))

Coefficients: [ -1.61934589 -10.0143165   21.62575245  17.7776278   -3.2764486
  -3.00851367  -9.97015572   4.54539824  25.08289881   2.82716666]
Intercept: [152.58395907]
Batch GD r2_score: 0.4985347446222317


In [18]:
import matplotlib.pyplot as plt
